# Baseline Transformer with BPE Tokenization

In [ ]:
!pip install -q datasets tqdm matplotlib tokenizers

In [ ]:
import math
import time
from dataclasses import dataclass
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from metrics import evaluate_all_metrics
from results_utils import save_experiment_artifacts


if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


@dataclass
class Config:
    train_stories: int = 5000
    val_stories: int = 2000
    tokenizer_train_stories: int = 50000
    vocab_size: int = 8000
    min_frequency: int = 2
    block_size: int = 128
    batch_size: int = 32
    d_model: int = 256
    n_heads: int = 4
    n_layers: int = 4
    dropout: float = 0.1
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    epochs: int = 3
    eval_every: int = 300
    val_max_batches: int = 20
    max_train_steps: Optional[int] = 15000
    sample_prompt: str = "Once upon a time"
    sample_tokens: int = 120


cfg = Config()
cfg

## Dataset Setup

In [ ]:
ds = load_dataset("roneneldan/TinyStories")
ds

In [ ]:
train_split = ds["train"].select(range(min(cfg.train_stories, len(ds["train"]))))
val_split = ds["validation"].select(range(min(cfg.val_stories, len(ds["validation"]))))
tokenizer_split = ds["train"].select(range(min(cfg.tokenizer_train_stories, len(ds["train"]))))

train_texts = train_split["text"]
val_texts = val_split["text"]
tokenizer_texts = tokenizer_split["text"]

print(f"Train stories: {len(train_texts):,}")
print(f"Validation stories: {len(val_texts):,}")
print(f"Tokenizer training stories: {len(tokenizer_texts):,}")
print(f"Avg train story chars: {sum(len(x) for x in train_texts) / len(train_texts):.1f}")

## BPE Tokenizer

In [ ]:
special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=cfg.vocab_size,
    min_frequency=cfg.min_frequency,
    special_tokens=special_tokens,
)
tokenizer.train_from_iterator(tokenizer_texts, trainer=trainer)

pad_token_id = tokenizer.token_to_id("[PAD]")
unk_token_id = tokenizer.token_to_id("[UNK]")
bos_token_id = tokenizer.token_to_id("[BOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")
vocab_size = tokenizer.get_vocab_size()


def encode(text):
    return tokenizer.encode(text).ids


def decode(token_ids):
    return tokenizer.decode([int(idx) for idx in token_ids], skip_special_tokens=True)


print(f"BPE vocab size: {vocab_size}")
print(tokenizer.encode(cfg.sample_prompt).tokens)


In [ ]:
def encode_story(text):
    return [bos_token_id] + encode(text) + [eos_token_id]


train_token_ids = [token_id for text in train_texts for token_id in encode_story(text)]
val_token_ids = [token_id for text in val_texts for token_id in encode_story(text)]

train_ids = torch.tensor(train_token_ids, dtype=torch.long)
val_ids = torch.tensor(val_token_ids, dtype=torch.long)

print(train_ids.shape, val_ids.shape)
print(f"Avg train story tokens: {sum(len(encode_story(text)) for text in train_texts) / len(train_texts):.1f}")

## Data Loader

In [ ]:
class BPEDataset(Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):
        x = self.token_ids[idx : idx + self.block_size]
        y = self.token_ids[idx + 1 : idx + self.block_size + 1]
        return x, y


train_dataset = BPEDataset(train_ids, cfg.block_size)
val_dataset = BPEDataset(val_ids, cfg.block_size)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

print(f"Train windows: {len(train_dataset):,}")
print(f"Validation windows: {len(val_dataset):,}")

## Baseline Transformer

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=128):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, block_size, block_size), persistent=False)

    def forward(self, x):
        batch_size, seq_len, channels = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_scores = attn_scores.masked_fill(~self.causal_mask[:, :, :seq_len, :seq_len], float("-inf"))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)
        out = self.out_proj(out)
        return self.resid_dropout(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=128):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout=dropout, block_size=block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class BaselineTransformer(nn.Module):
    def __init__(self, vocab_size, block_size, d_model, n_heads, n_layers, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, dropout=dropout, block_size=block_size) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x, targets=None):
        batch_size, seq_len = x.shape
        if seq_len > self.block_size:
            raise ValueError(f"Sequence length {seq_len} exceeds block size {self.block_size}")

        positions = torch.arange(seq_len, device=x.device)
        x = self.token_embedding(x) + self.position_embedding(positions)[None, :, :]
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(batch_size * seq_len, self.vocab_size), targets.reshape(batch_size * seq_len))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx


def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

## Training

In [ ]:
from experiment_utils import count_parameters, pick_device, train_model

device = pick_device()
print(f"Using device: {device}")

In [ ]:
model = BaselineTransformer(
    vocab_size=vocab_size,
    block_size=cfg.block_size,
    d_model=cfg.d_model,
    n_heads=cfg.n_heads,
    n_layers=cfg.n_layers,
    dropout=cfg.dropout,
).to(device)

print(f"Model parameters: {count_parameters(model) / 1e6:.2f}M")

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
    checkpoint_path="baseline_best.pt",
)

history["best_val_loss"]

In [ ]:
if history["steps"]:
    plt.figure(figsize=(10, 4))
    plt.plot(history["steps"], history["train_loss"], label="train loss")
    plt.plot(history["steps"], history["val_loss"], label="val loss")
    plt.xlabel("step")
    plt.ylabel("token cross-entropy")
    plt.title("BPE Transformer Training Curve")
    plt.legend()
    plt.show()
else:
    print("No intermediate evaluations were recorded. Lower cfg.eval_every if you want a curve.")

## Sample Generation

In [ ]:
prompt_ids = torch.tensor([[bos_token_id] + encode(cfg.sample_prompt)], dtype=torch.long, device=device)

model.load_state_dict(torch.load("baseline_best.pt", map_location=device))
sample_ids = model.generate(prompt_ids, max_new_tokens=cfg.sample_tokens, temperature=0.9)
sample_text = decode(sample_ids[0].tolist())
print(sample_text)

## Evaluation and Artifact Export

In [ ]:
ood_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
ood_texts = [text for text in ood_dataset["text"] if text.strip()][:200]


def byte_nll_fn(texts):
    model.eval()
    total_nll = 0.0
    total_bytes = 0

    with torch.no_grad():
        for text in texts:
            token_ids = [bos_token_id] + encode(text) + [eos_token_id]
            if len(token_ids) < 2:
                continue

            for start in range(0, len(token_ids) - 1, cfg.block_size):
                chunk = token_ids[start : start + cfg.block_size + 1]
                if len(chunk) < 2:
                    continue

                x = torch.tensor(chunk[:-1], dtype=torch.long, device=device)[None, :]
                y = torch.tensor(chunk[1:], dtype=torch.long, device=device)[None, :]
                logits, _ = model(x)
                nll = F.cross_entropy(
                    logits.reshape(-1, vocab_size),
                    y.reshape(-1),
                    reduction="sum",
                )

                total_nll += nll.item()

            total_bytes += len(text.encode("utf-8"))

    return total_nll / total_bytes


metrics = evaluate_all_metrics(
    in_domain_texts=val_texts,
    ood_texts=ood_texts,
    byte_nll_fn=byte_nll_fn,
    max_items=200,
    noisy_char_error_rate=0.05,
    noise_seed=42,
)
metrics

In [ ]:
save_experiment_artifacts(
    "baseline_bpe",
    metrics=metrics,
    config=cfg,
    sample_text=sample_text,
    extra={"ood_preview": ood_texts[:5]},
)

print("Saved baseline_bpe_metrics.json, baseline_bpe_config.json, baseline_bpe_sample.txt")